# Mini Project 2 — Part 3.1: Examine Commits Before and After the Gap

This notebook examines commits immediately before and after the longest inactivity gap for each of the ten assigned projects.

For each project, this notebook:
- identifies the longest inactivity gap;
- collects the last 10 commits before the gap;
- collects at least 10 commits after the gap, when available;
- labels each selected commit as `pre` or `post`;
- saves the selected commits to a combined CSV;
- classifies commit messages into the required themes;
- identifies the top two themes before and after the gap; and
- adds `BeforeThemes` and `AfterThemes` to the project statistics file.

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

NETID = "jmurph91"

# Part 1 input
PROJECT_SUMMARY_FILE = f"{NETID}_project_summary.csv"

# Part 2 statistics file
#PROJECT_STATS_FILE = f"{NETID}_project_stats.csv"
PROJECT_STATS_FILE = "part2_outputs/jmurph91_project_stats.csv"

# Keep Part 3.1 generated files together
OUTPUT_DIR = Path("part3_1_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Project summary: {PROJECT_SUMMARY_FILE}")
print(f"Project stats: {PROJECT_STATS_FILE}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")

Project summary: jmurph91_project_summary.csv
Project stats: part2_outputs/jmurph91_project_stats.csv
Output folder: /home/jmurph91/archeology/MiniProject2/MiniProject2/part3_1_outputs


In [23]:
df = pd.read_csv(
    PROJECT_SUMMARY_FILE,
    sep=";"
)

expected_columns = [
    "project_wocid",
    "commit_sha1",
    "author",
    "time",
    "commit message"
]

assert list(df.columns) == expected_columns, (
    f"Unexpected columns.\n"
    f"Expected: {expected_columns}\n"
    f"Found: {list(df.columns)}"
)

# Convert Unix timestamp to datetime
df["datetime"] = pd.to_datetime(
    df["time"],
    unit="s",
    errors="coerce"
)

assert df["datetime"].notna().all()

# Create monthly period
df["Month"] = df["datetime"].dt.to_period("M")

print(f"Loaded {len(df):,} commits.")
print(f"Projects: {df['project_wocid'].nunique()}")

Loaded 50,009 commits.
Projects: 10


In [24]:
project_stats = pd.read_csv(
    PROJECT_STATS_FILE,
    sep=";"
)

print("Loaded project statistics:")
print(project_stats.shape)
print()

print(project_stats.columns.tolist())

Loaded project statistics:
(10, 14)

['project', 'number of commits', 'number of authors', 'max time', 'min time', 'number of stars', 'number of forks', 'last commit date', 'LongestGapStart', 'LongestGapEnd', 'LongestGapLength', 'NumCommitsAfterLastGap', 'NumberOfGapsInTimeline', 'ActivityPattern']


In [25]:
required_gap_columns = [
    "project",
    "LongestGapStart",
    "LongestGapEnd",
    "LongestGapLength"
]

for column in required_gap_columns:
    assert column in project_stats.columns, (
        f"Missing required column: {column}"
    )

print("✓ Required Part 2 gap columns found.")
print()

print(
    project_stats[
        required_gap_columns
    ].to_string(index=False)
)

✓ Required Part 2 gap columns found.

                           project LongestGapStart LongestGapEnd  LongestGapLength
                    angel-ml_angel         2024-02       2025-05                16
                        bmclab_bmc         2016-12       2017-03                 4
                       brial_brial         2013-02       2015-04                27
        comit-network_xmr-btc-swap             NaN           NaN                 0
                 easystats_insight             NaN           NaN                 0
           jasp-stats_jasp-desktop         2013-11       2013-11                 1
juliarobotics_rigidbodydynamics.jl         2023-07       2024-09                15
                     mmorise_world         2022-03       2023-05                15
             samreay_chainconsumer         2022-10       2023-09                12
       systempiper_systempipeshiny         2023-11       2024-03                 5


In [26]:
def find_longest_gap(project_df):
    """
    Find the longest consecutive run of zero-commit months.
    
    Returns:
        start month
        end month
        length
    """

    counts = project_df.groupby("Month").size()

    first_month = counts.index.min()
    last_month = counts.index.max()

    all_months = pd.period_range(
        start=first_month,
        end=last_month,
        freq="M"
    )

    monthly = (
        counts
        .reindex(all_months, fill_value=0)
    )

    gaps = []

    start_index = None

    for i, count in enumerate(monthly):

        if count == 0 and start_index is None:
            start_index = i

        if (
            count != 0
            and start_index is not None
        ):
            end_index = i - 1

            gaps.append({
                "start": all_months[start_index],
                "end": all_months[end_index],
                "length": end_index - start_index + 1
            })

            start_index = None

    # Handle a gap ending at the final month
    if start_index is not None:

        end_index = len(all_months) - 1

        gaps.append({
            "start": all_months[start_index],
            "end": all_months[end_index],
            "length": end_index - start_index + 1
        })

    if not gaps:
        return None

    # Longest gap; earliest gap wins if tied
    longest = sorted(
        gaps,
        key=lambda x: (-x["length"], x["start"])
    )[0]

    return longest

In [27]:
projects = sorted(
    df["project_wocid"].unique()
)

gap_check = []

for project in projects:

    project_df = df[
        df["project_wocid"] == project
    ].copy()

    calculated_gap = find_longest_gap(
        project_df
    )

    stats_row = project_stats[
        project_stats["project"] == project
    ]

    assert len(stats_row) == 1

    stats_row = stats_row.iloc[0]

    if calculated_gap is None:

        calculated_start = ""
        calculated_end = ""
        calculated_length = 0

    else:

        calculated_start = str(
            calculated_gap["start"]
        )

        calculated_end = str(
            calculated_gap["end"]
        )

        calculated_length = calculated_gap[
            "length"
        ]

    gap_check.append({
        "project": project,
        "Part2_Start": str(
            stats_row["LongestGapStart"]
        ),
        "Calculated_Start": calculated_start,
        "Part2_End": str(
            stats_row["LongestGapEnd"]
        ),
        "Calculated_End": calculated_end,
        "Part2_Length": int(
            stats_row["LongestGapLength"]
        ),
        "Calculated_Length": calculated_length
    })

gap_check_df = pd.DataFrame(gap_check)

print(
    gap_check_df.to_string(index=False)
)

                           project Part2_Start Calculated_Start Part2_End Calculated_End  Part2_Length  Calculated_Length
                    angel-ml_angel     2024-02          2024-02   2025-05        2025-05            16                 16
                        bmclab_bmc     2016-12          2016-12   2017-03        2017-03             4                  4
                       brial_brial     2013-02          2013-02   2015-04        2015-04            27                 27
        comit-network_xmr-btc-swap         nan                        nan                            0                  0
                 easystats_insight         nan                        nan                            0                  0
           jasp-stats_jasp-desktop     2013-11          2013-11   2013-11        2013-11             1                  1
juliarobotics_rigidbodydynamics.jl     2023-07          2023-07   2024-09        2024-09            15                 15
                     mmo

In [28]:
for _, row in gap_check_df.iterrows():
    if row["Part2_Start"] != row["Calculated_Start"]:
        print(f"PROJECT: {row['project']}")
        print(f"Part 2 start:       {row['Part2_Start']!r}")
        print(f"Calculated start:   {row['Calculated_Start']!r}")
        print()
        
    if row["Part2_End"] != row["Calculated_End"]:
        print(f"PROJECT: {row['project']}")
        print(f"Part 2 end:         {row['Part2_End']!r}")
        print(f"Calculated end:     {row['Calculated_End']!r}")
        print()

PROJECT: comit-network_xmr-btc-swap
Part 2 start:       'nan'
Calculated start:   ''

PROJECT: comit-network_xmr-btc-swap
Part 2 end:         'nan'
Calculated end:     ''

PROJECT: easystats_insight
Part 2 start:       'nan'
Calculated start:   ''

PROJECT: easystats_insight
Part 2 end:         'nan'
Calculated end:     ''



In [29]:
def normalize_gap_value(value):
    if pd.isna(value):
        return ""
    
    value = str(value).strip()
    
    if value.lower() in {"nan", "none", ""}:
        return ""
    
    return value


for _, row in gap_check_df.iterrows():
    part2_start = normalize_gap_value(row["Part2_Start"])
    part2_end = normalize_gap_value(row["Part2_End"])
    
    calculated_start = normalize_gap_value(row["Calculated_Start"])
    calculated_end = normalize_gap_value(row["Calculated_End"])

    assert row["Part2_Length"] == row["Calculated_Length"], (
        f"Gap length mismatch for {row['project']}"
    )

    assert part2_start == calculated_start, (
        f"Gap start mismatch for {row['project']}: "
        f"Part 2 = {part2_start!r}, "
        f"Calculated = {calculated_start!r}"
    )

    assert part2_end == calculated_end, (
        f"Gap end mismatch for {row['project']}: "
        f"Part 2 = {part2_end!r}, "
        f"Calculated = {calculated_end!r}"
    )

print("✓ All Part 2 longest gaps match the recalculated gaps.")

✓ All Part 2 longest gaps match the recalculated gaps.


In [30]:
gap_commits = []

for project in projects:

    project_df = df[
        df["project_wocid"] == project
    ].copy()

    project_df = project_df.sort_values(
        "datetime"
    ).reset_index(drop=True)

    gap = find_longest_gap(project_df)

    # --------------------------------------------------------
    # No inactivity gap
    # --------------------------------------------------------

    if gap is None:

        print(
            f"{project}: no inactivity gap found; "
            f"skipping pre/post selection."
        )

        continue

    gap_start = gap["start"]
    gap_end = gap["end"]

    # --------------------------------------------------------
    # Commits before the gap
    # --------------------------------------------------------

    before = project_df[
        project_df["Month"] < gap_start
    ].copy()

    before = before.tail(10)

    before["pre/post"] = "pre"

    # --------------------------------------------------------
    # Commits after the gap
    # --------------------------------------------------------

    after = project_df[
        project_df["Month"] > gap_end
    ].copy()

    # At least 10 when available
    after = after.head(10)

    after["pre/post"] = "post"

    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    selected = pd.concat(
        [before, after],
        ignore_index=True
    )

    # Keep original project summary columns
    selected = selected[
        [
            "pre/post",
            "project_wocid",
            "commit_sha1",
            "author",
            "time",
            "commit message"
        ]
    ]

    gap_commits.append(selected)

    print(
        f"{project}: "
        f"{len(before)} pre commits + "
        f"{len(after)} post commits"
    )

# Combine all projects
gap_commits_df = pd.concat(
    gap_commits,
    ignore_index=True
)

print()
print(
    f"Total selected commits: "
    f"{len(gap_commits_df):,}"
)

angel-ml_angel: 10 pre commits + 10 post commits
bmclab_bmc: 10 pre commits + 10 post commits
brial_brial: 10 pre commits + 10 post commits
comit-network_xmr-btc-swap: no inactivity gap found; skipping pre/post selection.
easystats_insight: no inactivity gap found; skipping pre/post selection.
jasp-stats_jasp-desktop: 10 pre commits + 10 post commits
juliarobotics_rigidbodydynamics.jl: 10 pre commits + 10 post commits
mmorise_world: 10 pre commits + 10 post commits
samreay_chainconsumer: 10 pre commits + 10 post commits
systempiper_systempipeshiny: 10 pre commits + 10 post commits

Total selected commits: 160


In [31]:
print(
    gap_commits_df[
        [
            "pre/post",
            "project_wocid",
            "commit_sha1",
            "author",
            "time",
            "commit message"
        ]
    ].head(20).to_string(index=False)
)

pre/post  project_wocid                              commit_sha1                                                              author       time                                                                                                                       commit message
     pre angel-ml_angel 3fc0c9c72426a772d9d63466c54f5e1f5fa5b815                                    Ouyang Wen <ouyangwen21@163.com> 1669277187                                                     Merge pull request #1256 from jyswpp/lcc_dev_part1\n\nlcc algo development part1
     pre angel-ml_angel e53cd11be56e2aeff89ef443ae61248fa8c4a724                                    Ouyang Wen <ouyangwen21@163.com> 1669277274                                                 Merge pull request #1255 from jyswpp/lcc_cluster_example2\n\nadd lcc cluster example
     pre angel-ml_angel 7dbccf90a34e44bc0df305de3f9ba240fc4be5e8       YinHan-Zhang <89371343+YinHan-Zhang@users.noreply.github.com> 1669951370                          

In [32]:
gap_commits_file = (
    OUTPUT_DIR
    / f"{NETID}_project_gap_commits.csv"
)

gap_commits_df.to_csv(
    gap_commits_file,
    index=False,
    sep=";"
)

print("✓ Gap commit file saved:")
print(gap_commits_file)

✓ Gap commit file saved:
part3_1_outputs/jmurph91_project_gap_commits.csv


In [33]:
def classify_theme(message):
    """
    Classify a commit message into one of the required
    Part 3.1 themes.
    """

    if pd.isna(message):
        return "Other"

    message = str(message).strip().lower()

    # --------------------------------------------------------
    # Automated bot contributions
    # --------------------------------------------------------

    bot_words = [
        "dependabot",
        "renovate",
        "github actions",
        "github-actions",
        "bot",
        "automated",
        "auto-update",
        "autoupdate",
        "ci:",
        "ci ",
        "formatting",
        "prettier",
        "black formatting",
        "lint"
    ]

    if any(word in message for word in bot_words):
        return "Automated bot contributions"

    # --------------------------------------------------------
    # Security patches
    # --------------------------------------------------------

    security_words = [
        "security",
        "vulnerability",
        "vulnerabilities",
        "cve-",
        "exploit",
        "xss",
        "csrf",
        "authentication vulnerability",
        "security patch"
    ]

    if any(word in message for word in security_words):
        return "Security patches"

    # --------------------------------------------------------
    # Dependency updates
    # --------------------------------------------------------

    dependency_words = [
        "dependenc",
        "dependency",
        "dependencies",
        "upgrade ",
        "update ",
        "bump ",
        "package",
        "packages",
        "requirements",
        "npm ",
        "pip ",
        "maven",
        "gradle"
    ]

    if any(word in message for word in dependency_words):
        return "Dependency updates"

    # --------------------------------------------------------
    # Release
    # --------------------------------------------------------

    release_words = [
        "release",
        "version ",
        "version:",
        "v1.",
        "v2.",
        "v3.",
        "v4.",
        "v5.",
        "publish",
        "published",
        "tag "
    ]

    if any(word in message for word in release_words):
        return "Release"

    # --------------------------------------------------------
    # Documentation
    # --------------------------------------------------------

    documentation_words = [
        "documentation",
        "document",
        "docs",
        "readme",
        "read me",
        "wiki",
        "tutorial",
        "comment",
        "comments",
        "typo"
    ]

    if any(word in message for word in documentation_words):
        return "Documentation updates"

    # --------------------------------------------------------
    # Bug fixes
    # --------------------------------------------------------

    bug_words = [
        "fix",
        "fixed",
        "fixes",
        "bug",
        "bugs",
        "issue",
        "error",
        "crash",
        "broken",
        "failure",
        "correct",
        "correction",
        "patch"
    ]

    if any(word in message for word in bug_words):
        return "Bug fixes"

    # --------------------------------------------------------
    # Feature development
    # --------------------------------------------------------

    feature_words = [
        "feature",
        "feat",
        "add ",
        "added",
        "implement",
        "implemented",
        "implementation",
        "support",
        "new ",
        "introduce",
        "create",
        "develop"
    ]

    if any(word in message for word in feature_words):
        return "Feature development"

    # --------------------------------------------------------
    # Everything else
    # --------------------------------------------------------

    return "Other"

In [34]:
gap_commits_df["Theme"] = (
    gap_commits_df["commit message"]
    .apply(classify_theme)
)

print(
    gap_commits_df[
        [
            "project_wocid",
            "pre/post",
            "Theme",
            "commit message"
        ]
    ].head(30).to_string(index=False)
)

 project_wocid pre/post               Theme                                                                                                                       commit message
angel-ml_angel      pre Feature development                                                     Merge pull request #1256 from jyswpp/lcc_dev_part1\n\nlcc algo development part1
angel-ml_angel      pre Feature development                                                 Merge pull request #1255 from jyswpp/lcc_cluster_example2\n\nadd lcc cluster example
angel-ml_angel      pre               Other                                       Merge a78cae13a19a440033326145837d9c65f86117c3 into e53cd11be56e2aeff89ef443ae61248fa8c4a724\n
angel-ml_angel      pre               Other                                       Merge 4cfe0d0d272f21b40a1e60e9bb216b98d42e951b into e53cd11be56e2aeff89ef443ae61248fa8c4a724\n
angel-ml_angel      pre               Other                                       Merge c680e55319eed38857e68c26432

In [35]:
theme_counts = (
    gap_commits_df
    .groupby(["project_wocid", "pre/post", "Theme"])
    .size()
    .reset_index(name="Count")
)

print(
    theme_counts.to_string(index=False)
)

                     project_wocid pre/post                       Theme  Count
                    angel-ml_angel     post                   Bug fixes      4
                    angel-ml_angel     post         Feature development      6
                    angel-ml_angel      pre                   Bug fixes      1
                    angel-ml_angel      pre         Feature development      2
                    angel-ml_angel      pre                       Other      7
                        bmclab_bmc     post                       Other     10
                        bmclab_bmc      pre                       Other     10
                       brial_brial     post                   Bug fixes      2
                       brial_brial     post          Dependency updates      1
                       brial_brial     post         Feature development      4
                       brial_brial     post                       Other      2
                       brial_brial     post         

In [36]:
def get_top_two_themes(project, phase):
    """
    Return the two most common themes for a project
    during either the pre-gap or post-gap period.
    """

    subset = gap_commits_df[
        (gap_commits_df["project_wocid"] == project)
        &
        (gap_commits_df["pre/post"] == phase)
    ]

    if subset.empty:
        return "Other:Other"

    counts = (
        subset["Theme"]
        .value_counts()
    )

    # Take the top two
    top = counts.head(2).index.tolist()

    # If only one theme exists, fill the second position
    if len(top) == 1:
        top.append("Other")

    return ":".join(top)

In [37]:
theme_records = []

for project in projects:

    before_themes = get_top_two_themes(
        project,
        "pre"
    )

    after_themes = get_top_two_themes(
        project,
        "post"
    )

    theme_records.append({
        "project": project,
        "BeforeThemes": before_themes,
        "AfterThemes": after_themes
    })

theme_df = pd.DataFrame(theme_records)

print(
    theme_df.to_string(index=False)
)

                           project                 BeforeThemes                   AfterThemes
                    angel-ml_angel    Other:Feature development Feature development:Bug fixes
                        bmclab_bmc                  Other:Other                   Other:Other
                       brial_brial                Other:Release     Feature development:Other
        comit-network_xmr-btc-swap                  Other:Other                   Other:Other
                 easystats_insight                  Other:Other                   Other:Other
           jasp-stats_jasp-desktop              Other:Bug fixes               Other:Bug fixes
juliarobotics_rigidbodydynamics.jl     Other:Dependency updates      Other:Dependency updates
                     mmorise_world    Feature development:Other  Dependency updates:Bug fixes
             samreay_chainconsumer              Other:Bug fixes               Other:Bug fixes
       systempiper_systempipeshiny Dependency updates:Bug fi

In [38]:
# Remove these columns if they already exist,
# so rerunning the cell does not create duplicates.

project_stats = project_stats.drop(
    columns=[
        "BeforeThemes",
        "AfterThemes"
    ],
    errors="ignore"
)

project_stats = project_stats.merge(
    theme_df,
    on="project",
    how="left"
)

print(
    project_stats[
        [
            "project",
            "BeforeThemes",
            "AfterThemes"
        ]
    ].to_string(index=False)
)

                           project                 BeforeThemes                   AfterThemes
                    angel-ml_angel    Other:Feature development Feature development:Bug fixes
                        bmclab_bmc                  Other:Other                   Other:Other
                       brial_brial                Other:Release     Feature development:Other
        comit-network_xmr-btc-swap                  Other:Other                   Other:Other
                 easystats_insight                  Other:Other                   Other:Other
           jasp-stats_jasp-desktop              Other:Bug fixes               Other:Bug fixes
juliarobotics_rigidbodydynamics.jl     Other:Dependency updates      Other:Dependency updates
                     mmorise_world    Feature development:Other  Dependency updates:Bug fixes
             samreay_chainconsumer              Other:Bug fixes               Other:Bug fixes
       systempiper_systempipeshiny Dependency updates:Bug fi

In [39]:
updated_stats_file = (
    OUTPUT_DIR
    / f"{NETID}_project_stats.csv"
)

project_stats.to_csv(
    updated_stats_file,
    index=False,
    sep=";"
)

print("✓ Updated project statistics saved:")
print(updated_stats_file)

✓ Updated project statistics saved:
part3_1_outputs/jmurph91_project_stats.csv


In [40]:
# Verify gap commit columns
expected_gap_columns = [
    "pre/post",
    "project_wocid",
    "commit_sha1",
    "author",
    "time",
    "commit message",
    "Theme"
]

assert list(gap_commits_df.columns) == expected_gap_columns

# Verify required stats columns
assert "BeforeThemes" in project_stats.columns
assert "AfterThemes" in project_stats.columns

# Verify all projects are represented
assert set(project_stats["project"]) == set(projects)

# Verify pre/post values
assert set(
    gap_commits_df["pre/post"].unique()
).issubset({"pre", "post"})

print("✓ Gap commit columns verified")
print("✓ BeforeThemes verified")
print("✓ AfterThemes verified")
print("✓ All projects represented")
print("✓ Pre/post labels verified")

✓ Gap commit columns verified
✓ BeforeThemes verified
✓ AfterThemes verified
✓ All projects represented
✓ Pre/post labels verified


In [41]:
print("=" * 70)
print("PART 3.1 RESULTS")
print("=" * 70)

print()

print("Gap commits:")
print(
    gap_commits_file
)

print()

print("Updated project statistics:")
print(
    updated_stats_file
)

print()

print("Theme summary:")
print(
    project_stats[
        [
            "project",
            "BeforeThemes",
            "AfterThemes"
        ]
    ].to_string(index=False)
)

print()

print("Output folder:")
print(
    OUTPUT_DIR.resolve()
)

PART 3.1 RESULTS

Gap commits:
part3_1_outputs/jmurph91_project_gap_commits.csv

Updated project statistics:
part3_1_outputs/jmurph91_project_stats.csv

Theme summary:
                           project                 BeforeThemes                   AfterThemes
                    angel-ml_angel    Other:Feature development Feature development:Bug fixes
                        bmclab_bmc                  Other:Other                   Other:Other
                       brial_brial                Other:Release     Feature development:Other
        comit-network_xmr-btc-swap                  Other:Other                   Other:Other
                 easystats_insight                  Other:Other                   Other:Other
           jasp-stats_jasp-desktop              Other:Bug fixes               Other:Bug fixes
juliarobotics_rigidbodydynamics.jl     Other:Dependency updates      Other:Dependency updates
                     mmorise_world    Feature development:Other  Dependency upda